# WRDS Financial Analysis Tool
**Track 2 — GitHub Data Analysis Project**

Pipeline: Data Collection → Data Cleaning → Data Analysis → Result Output

Compares three U.S. companies against their industry peers using Compustat (WRDS).

---

## Section 1 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import wrds

## Section 2 — Function Definitions

All helper and analysis functions are defined here, grouped by purpose:
- **2.1** WRDS connection & schema utilities
- **2.2** Data collection (company + industry)
- **2.3** Data cleaning & financial ratio computation
- **2.4** Visualization

### 2.1 — WRDS Connection & Schema Utilities

In [ ]:
def connect_wrds():
    """Open a WRDS database connection."""
    return wrds.Connection()


def get_table_columns(db, schema, table):
    """Return the column list for a given WRDS schema.table."""
    sql = f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = '{schema}'
          AND table_name   = '{table}'
        ORDER BY ordinal_position
    """
    return db.raw_sql(sql)["column_name"].tolist()


def pick_col(columns, candidates):
    """Return the first candidate column that exists in `columns`."""
    for c in candidates:
        if c in columns:
            return c
    return None


def clean_tickers(raw_input):
    """Validate and normalise a comma-separated list of exactly 3 tickers."""
    tickers = [t.strip().upper() for t in raw_input.split(",") if t.strip()]
    if len(tickers) != 3:
        raise ValueError("Please enter exactly 3 tickers separated by commas, e.g.  NKE,LULU,UA")
    return tickers


def validate_dates(start, end):
    """Parse and validate that start_date <= end_date."""
    s = pd.to_datetime(start, format="%Y-%m-%d", errors="raise")
    e = pd.to_datetime(end,   format="%Y-%m-%d", errors="raise")
    if s > e:
        raise ValueError("start_date must be earlier than or equal to end_date.")
    return start, end

### 2.2 — Data Collection

In [ ]:
def fetch_company_raw(db, tickers, start_date, end_date):
    """
    Pull annual Compustat (comp.funda) data for the requested tickers.

    Fields retrieved (all monetary values in millions USD):
      gvkey, datadate, tic
      at   — total assets
      sale — net sales / revenue
      ni   — net income
      ceq  — common/ordinary equity
    """
    funda_cols = get_table_columns(db, "comp", "funda")

    tic_col    = pick_col(funda_cols, ["tic", "ticker"])
    date_col   = pick_col(funda_cols, ["datadate"])
    gvkey_col  = pick_col(funda_cols, ["gvkey"])
    at_col     = pick_col(funda_cols, ["at"])
    sale_col   = pick_col(funda_cols, ["sale", "revt"])
    ni_col     = pick_col(funda_cols, ["ni", "ib"])
    ceq_col    = pick_col(funda_cols, ["ceq"])

    for col, name in [(tic_col, "tic"), (date_col, "datadate"), (gvkey_col, "gvkey"),
                      (at_col, "at"), (sale_col, "sale"), (ni_col, "ni"), (ceq_col, "ceq")]:
        if col is None:
            raise ValueError(f"Required field '{name}' not found in comp.funda.")

    ticker_list = "', '".join(tickers)
    sql = f"""
        SELECT
            {gvkey_col}  AS gvkey,
            {date_col}   AS datadate,
            {tic_col}    AS tic,
            {at_col}     AS at,
            {sale_col}   AS sale,
            {ni_col}     AS ni,
            {ceq_col}    AS ceq
        FROM comp.funda
        WHERE {tic_col}  IN ('{ticker_list}')
          AND {date_col} BETWEEN '{start_date}' AND '{end_date}'
          AND indfmt = 'INDL'
          AND datafmt = 'STD'
          AND popsrc = 'D'
          AND consol = 'C'
        ORDER BY {tic_col}, {date_col}
    """
    df = db.raw_sql(sql)
    if df.empty:
        raise ValueError("No company data returned. Check your tickers and date range.")
    return df


def fetch_industry_code(db, gvkeys):
    """
    Retrieve the SIC code(s) for the selected companies from comp.company.
    Returns (industry_field_name, most_common_sic_code).
    """
    company_cols = get_table_columns(db, "comp", "company")
    gvkey_col    = pick_col(company_cols, ["gvkey"])
    ind_field    = pick_col(company_cols, ["sic", "sich", "gind", "ggroup", "gsector"])

    if gvkey_col is None or ind_field is None:
        raise ValueError("comp.company does not expose a usable gvkey/industry field.")

    gvkey_list = "', '".join(gvkeys)
    sql = f"""
        SELECT {gvkey_col} AS gvkey, {ind_field} AS ind_code
        FROM comp.company
        WHERE {gvkey_col} IN ('{gvkey_list}')
    """
    df = db.raw_sql(sql).dropna(subset=["ind_code"])
    if df.empty:
        raise ValueError("No industry code found for the selected companies.")

    modal_code = df["ind_code"].mode().iloc[0]
    return ind_field, modal_code


def fetch_industry_raw(db, ind_field, ind_code, start_date, end_date):
    """
    Pull annual Compustat data for all companies sharing the same SIC/industry code,
    then compute per-company, per-year financial ratios and aggregate to yearly averages.

    Strategy:
      1. Join comp.funda with comp.company on gvkey to filter by ind_code.
      2. Compute individual-firm ratios first (avoids ratio-of-averages bias).
      3. Return year-level mean ratios across all industry peers.
    """
    funda_cols   = get_table_columns(db, "comp", "funda")
    company_cols = get_table_columns(db, "comp", "company")

    f_gvkey  = pick_col(funda_cols, ["gvkey"])
    f_date   = pick_col(funda_cols, ["datadate"])
    f_at     = pick_col(funda_cols, ["at"])
    f_sale   = pick_col(funda_cols, ["sale", "revt"])
    f_ni     = pick_col(funda_cols, ["ni", "ib"])
    f_ceq    = pick_col(funda_cols, ["ceq"])
    c_gvkey  = pick_col(company_cols, ["gvkey"])

    if ind_field not in company_cols:
        raise ValueError(f"Industry field '{ind_field}' missing from comp.company.")

    code_expr = f"'{ind_code}'" if isinstance(ind_code, str) else str(int(ind_code))

    sql = f"""
        SELECT
            f.{f_gvkey}  AS gvkey,
            f.{f_date}   AS datadate,
            f.{f_at}     AS at,
            f.{f_sale}   AS sale,
            f.{f_ni}     AS ni,
            f.{f_ceq}    AS ceq
        FROM comp.funda f
        INNER JOIN comp.company c
            ON f.{f_gvkey} = c.{c_gvkey}
        WHERE c.{ind_field} = {code_expr}
          AND f.{f_date} BETWEEN '{start_date}' AND '{end_date}'
          AND f.indfmt  = 'INDL'
          AND f.datafmt = 'STD'
          AND f.popsrc  = 'D'
          AND f.consol  = 'C'
        ORDER BY f.{f_gvkey}, f.{f_date}
    """
    df = db.raw_sql(sql)
    if df.empty:
        raise ValueError("No industry peer data returned for the detected industry code.")
    return df

### 2.3 — Data Cleaning & Financial Ratio Computation

In [ ]:
def clean_raw(df, label="data"):
    """
    Step 1 — Data Cleaning:
      - Parse datatypes
      - Drop rows missing critical fields (at, sale, ni, ceq)
      - Remove duplicates (keep latest datadate per gvkey-year)
      - Filter out firms with non-positive total assets or equity
    """
    df = df.copy()
    df["datadate"] = pd.to_datetime(df["datadate"])
    df["year"]     = df["datadate"].dt.year

    if "tic" in df.columns:
        df["tic"] = df["tic"].astype(str).str.upper().str.strip()
    if "gvkey" in df.columns:
        df["gvkey"] = df["gvkey"].astype(str).str.strip()

    before = len(df)
    df.dropna(subset=["at", "sale", "ni", "ceq"], inplace=True)
    df = df[(df["at"] > 0) & (df["ceq"] != 0) & (df["sale"] > 0)]

    # Keep one observation per entity-year (latest date)
    if "tic" in df.columns:
        df = df.sort_values("datadate").drop_duplicates(subset=["tic", "year"], keep="last")
    else:
        df = df.sort_values("datadate").drop_duplicates(subset=["gvkey", "year"], keep="last")

    after = len(df)
    print(f"  [{label}] rows before cleaning: {before}  →  after: {after}")
    return df.reset_index(drop=True)


def compute_ratios(df, group_col="tic"):
    """
    Step 2 — Financial Ratio Computation:
      ROE              = Net Income / Common Equity          (%, return on equity)
      ROA              = Net Income / Total Assets           (%, return on assets)
      Profit Margin    = Net Income / Revenue                (%)
      Asset Turnover   = Revenue    / Total Assets           (times)
      Revenue Growth   = (Revenue_t - Revenue_{t-1}) / Revenue_{t-1}  (%)

    Revenue Growth is computed within each group (company or peer firm).
    Rows with no prior-year observation receive NaN for Revenue Growth.
    """
    df = df.sort_values([group_col, "year"]).copy()

    df["roe"]              = df["ni"]   / df["ceq"]  * 100
    df["roa"]              = df["ni"]   / df["at"]   * 100
    df["profit_margin"]    = df["ni"]   / df["sale"] * 100
    df["asset_turnover"]   = df["sale"] / df["at"]
    df["rev_growth"]       = df.groupby(group_col)["sale"].pct_change() * 100

    # Winsorise extreme ratio values (1st–99th percentile) to reduce outlier distortion
    ratio_cols = ["roe", "roa", "profit_margin", "asset_turnover", "rev_growth"]
    for col in ratio_cols:
        lo, hi = df[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(lo, hi)

    return df


def aggregate_company(df):
    """
    Step 3a — Company-level summary: mean of each ratio across all years.
    """
    ratio_cols = ["roe", "roa", "profit_margin", "asset_turnover", "rev_growth",
                  "at", "sale", "ni"]
    return df.groupby("tic")[ratio_cols].mean().reset_index()


def aggregate_industry(df):
    """
    Step 3b — Industry summary: per-firm ratios computed first, then yearly medians
    (median is more robust to skewed distributions from large conglomerates).
    Returns both a per-year table and an overall scalar summary row.
    """
    ratio_cols = ["roe", "roa", "profit_margin", "asset_turnover", "rev_growth",
                  "at", "sale", "ni"]

    yearly = df.groupby("year")[ratio_cols].median().reset_index()

    summary_row = yearly[ratio_cols].mean().to_frame().T
    summary_row["tic"] = "Industry Avg"
    return yearly, summary_row


def build_summary_table(company_summary, industry_summary_row):
    """Combine company-level and industry-average rows into one display table."""
    cols = ["tic", "roe", "roa", "profit_margin", "asset_turnover", "rev_growth",
            "at", "sale", "ni"]
    return pd.concat(
        [company_summary[cols], industry_summary_row[cols]],
        ignore_index=True
    )

### 2.4 — Visualization Functions

In [ ]:
COLORS = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]  # company 1/2/3, industry


def _fmt_millions(ax, axis="y"):
    """Format an axis tick label to show values in millions USD."""
    import matplotlib.ticker as mticker
    fmt = mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M")
    if axis == "y":
        ax.yaxis.set_major_formatter(fmt)
    else:
        ax.xaxis.set_major_formatter(fmt)


# ── Chart 1: Line Chart ────────────────────────────────────────────────────────
def plot_line_chart(company_df, industry_yearly, industry_name):
    """
    Revenue (Sales) trend over time.
    X-axis: fiscal year (integer)  |  Y-axis: Net Sales (USD millions)
    """
    fig, ax = plt.subplots(figsize=(11, 6))

    tickers = company_df["tic"].unique()
    for i, tic in enumerate(tickers):
        sub = company_df[company_df["tic"] == tic].sort_values("year")
        ax.plot(sub["year"].astype(int), sub["sale"], marker="o", color=COLORS[i], label=tic, linewidth=2)

    ind = industry_yearly.sort_values("year")
    ax.plot(ind["year"].astype(int), ind["sale"], marker="s", color=COLORS[3],
            linestyle="--", linewidth=2, label=f"{industry_name} Industry Median")

    # Force integer-only year ticks
    all_years = sorted(company_df["year"].astype(int).unique())
    ax.set_xticks(all_years)
    ax.set_xticklabels(all_years, rotation=45, ha="right")

    ax.set_title("Revenue (Net Sales) Over Time", fontsize=14, fontweight="bold")
    ax.set_xlabel("Fiscal Year", fontsize=12)
    ax.set_ylabel("Net Sales (USD millions)", fontsize=12)
    _fmt_millions(ax, "y")
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


# ── Chart 2: Scatter Chart ─────────────────────────────────────────────────────
def plot_scatter_chart(company_df, industry_yearly, industry_name):
    """
    ROA vs ROE scatter — profitability efficiency quadrant.
    X-axis: ROA (%)  |  Y-axis: ROE (%)
    Point size encodes absolute revenue scale.
    """
    fig, ax = plt.subplots(figsize=(9, 7))

    tickers = company_df["tic"].unique()
    for i, tic in enumerate(tickers):
        sub = company_df[company_df["tic"] == tic]
        size = np.clip(sub["sale"] / sub["sale"].max() * 300, 30, 300)
        ax.scatter(sub["roa"], sub["roe"], s=size, color=COLORS[i], alpha=0.75,
                   label=tic, edgecolors="white", linewidth=0.5)
        for _, row in sub.iterrows():
            ax.annotate(str(int(row["year"])), (row["roa"], row["roe"]),
                        textcoords="offset points", xytext=(5, 3), fontsize=7, color=COLORS[i])

    ind = industry_yearly
    ax.scatter(ind["roa"], ind["roe"], s=120, color=COLORS[3], marker="D",
               label=f"{industry_name} Industry Median", zorder=5, edgecolors="black", linewidth=0.8)

    ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.set_title("ROA vs. ROE by Year (bubble size ∝ revenue)", fontsize=14, fontweight="bold")
    ax.set_xlabel("Return on Assets — ROA (%)", fontsize=12)
    ax.set_ylabel("Return on Equity — ROE (%)", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()


# ── Chart 3: Bar Chart ─────────────────────────────────────────────────────────
def plot_bar_chart(summary_df, industry_name):
    """
    Grouped bar chart: ROE, ROA, Profit Margin side-by-side for each entity.
    Y-axis: Ratio (%)
    """
    entities = summary_df["tic"].tolist()
    metrics  = ["roe", "roa", "profit_margin"]
    labels   = ["ROE (%)", "ROA (%)", "Profit Margin (%)"]

    x     = np.arange(len(entities))
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 6))
    for j, (metric, label) in enumerate(zip(metrics, labels)):
        vals = summary_df[metric].values
        bars = ax.bar(x + (j - 1) * width, vals, width=width * 0.9, label=label)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.3, f"{v:.1f}%",
                    ha="center", va="bottom", fontsize=7.5)

    ax.set_title("Average Profitability Ratios: Companies vs Industry", fontsize=14, fontweight="bold")
    ax.set_xlabel("Entity", fontsize=12)
    ax.set_ylabel("Ratio (%)", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(entities, fontsize=11)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


# ── Chart 4: Box Chart ─────────────────────────────────────────────────────────
def plot_box_chart(company_df, industry_name):
    """
    Box plots showing the year-by-year distribution of four key ratios
    for each of the three companies.
    """
    ratio_info = [
        ("roe",            "ROE (%)"),
        ("roa",            "ROA (%)"),
        ("profit_margin",  "Profit Margin (%)"),
        ("asset_turnover", "Asset Turnover (times)"),
    ]

    tickers = company_df["tic"].unique()
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax, (col, ylabel) in zip(axes, ratio_info):
        data   = [company_df.loc[company_df["tic"] == t, col].dropna().values for t in tickers]
        bp     = ax.boxplot(data, patch_artist=True, notch=False,
                            medianprops={"color": "black", "linewidth": 2})
        for patch, color in zip(bp["boxes"], COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax.set_xticklabels(tickers, fontsize=10)
        ax.set_title(ylabel, fontsize=12, fontweight="bold")
        ax.set_ylabel(ylabel, fontsize=10)
        ax.grid(axis="y", linestyle="--", alpha=0.4)

    fig.suptitle("Distribution of Financial Ratios by Company (annual observations)",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


# ── Chart 5: Radar Chart ───────────────────────────────────────────────────────
def plot_radar_chart(summary_df, industry_name):
    """
    Spider / radar chart comparing five financial dimensions (normalised 0–1):
      ROE (%)  |  ROA (%)  |  Profit Margin (%)  |  Asset Turnover (×)  |  Revenue Growth (%)
    """
    metrics = ["roe", "roa", "profit_margin", "asset_turnover", "rev_growth"]
    metric_labels = ["ROE (%)", "ROA (%)", "Profit Margin (%)",
                     "Asset Turnover (×)", "Revenue Growth (%)"]

    plot_df = summary_df.set_index("tic")[metrics].copy()

    for col in metrics:
        lo, hi = plot_df[col].min(), plot_df[col].max()
        plot_df[col] = 0.5 if lo == hi else (plot_df[col] - lo) / (hi - lo)

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})

    for i, entity in enumerate(plot_df.index):
        vals = plot_df.loc[entity, metrics].tolist() + [plot_df.loc[entity, metrics[0]]]
        color = COLORS[i] if i < len(COLORS) else "#607D8B"
        ax.plot(angles, vals, color=color, linewidth=2, label=entity)
        ax.fill(angles, vals, color=color, alpha=0.10)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=8, color="grey")
    ax.set_title("Radar Chart — Five Financial Dimensions (normalised)",
                 fontsize=13, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
    plt.tight_layout()
    plt.show()

---
## Section 3 — User Input

Run this cell and follow the prompts.  
Examples:
- Tickers: `NKE,LULU,UAA`
- Industry name: `Sportswear & Apparel`
- Start date: `2015-01-01`
- End date: `2023-12-31`

In [ ]:
# ── User Input ─────────────────────────────────────────────────────────────────
raw_tickers   = input("Enter 3 US company tickers (comma-separated, e.g. NKE,LULU,UAA): ")
industry_name = input("Enter the industry name for display (e.g. Sportswear & Apparel): ").strip()
start_date    = input("Enter start date (YYYY-MM-DD): ").strip()
end_date      = input("Enter end date   (YYYY-MM-DD): ").strip()

tickers = clean_tickers(raw_tickers)
start_date, end_date = validate_dates(start_date, end_date)

print(f"\n  Tickers      : {tickers}")
print(f"  Industry     : {industry_name}")
print(f"  Date range   : {start_date}  →  {end_date}")

---
## Section 4 — Main Execution & Output

### 4.1 — Data Collection

In [ ]:
print("[Step 1] Connecting to WRDS ...")
db = connect_wrds()

print("[Step 2] Fetching company data from comp.funda ...")
company_raw = fetch_company_raw(db, tickers, start_date, end_date)
print(f"  Raw rows retrieved: {len(company_raw)}")
display(
    company_raw.head(6).rename(columns={
        "at":   "Total Assets (USD M)",
        "sale": "Net Sales (USD M)",
        "ni":   "Net Income (USD M)",
        "ceq":  "Common Equity (USD M)",
    })
)

print("[Step 3] Detecting industry code from comp.company ...")
gvkeys = company_raw["gvkey"].dropna().astype(str).unique().tolist()
ind_field, ind_code = fetch_industry_code(db, gvkeys)
print(f"  Industry field : {ind_field}")
print(f"  Industry code  : {ind_code}  (SIC or equivalent)")

print("[Step 4] Fetching industry peer data ...")
industry_raw = fetch_industry_raw(db, ind_field, ind_code, start_date, end_date)
print(f"  Industry peer rows: {len(industry_raw)}")
display(
    industry_raw.head(6).rename(columns={
        "at":   "Total Assets (USD M)",
        "sale": "Net Sales (USD M)",
        "ni":   "Net Income (USD M)",
        "ceq":  "Common Equity (USD M)",
    })
)

### 4.2 — Data Cleaning

In [ ]:
print("[Step 5] Cleaning company data ...")
company_clean = clean_raw(company_raw, label="companies")

print("[Step 6] Cleaning industry peer data ...")
industry_clean = clean_raw(industry_raw, label="industry peers")

col_units = {
    "at":   "Total Assets (USD M)",
    "sale": "Net Sales (USD M)",
    "ni":   "Net Income (USD M)",
    "ceq":  "Common Equity (USD M)",
}

print("\nCompany data (cleaned):")
display(
    company_clean[["tic", "year", "at", "sale", "ni", "ceq"]]
    .rename(columns=col_units)
    .head(9)
)
print("\nIndustry peer data (cleaned, first 6 rows):")
display(
    industry_clean[["gvkey", "year", "at", "sale", "ni", "ceq"]]
    .rename(columns=col_units)
    .head(6)
)

### 4.3 — Financial Ratio Analysis

In [ ]:
print("[Step 7] Computing financial ratios for companies ...")
company_ratios = compute_ratios(company_clean, group_col="tic")

print("[Step 8] Computing financial ratios for industry peers ...")
industry_ratios = compute_ratios(industry_clean, group_col="gvkey")

print("[Step 9] Aggregating to summary tables ...")
company_summary          = aggregate_company(company_ratios)
industry_yearly, ind_row = aggregate_industry(industry_ratios)
summary_df               = build_summary_table(company_summary, ind_row)

ratio_display_cols = ["tic", "roe", "roa", "profit_margin", "asset_turnover", "rev_growth"]
print("\n=== Summary: Mean Financial Ratios ===")
display(
    summary_df[ratio_display_cols]
    .rename(columns={
        "tic": "Entity",
        "roe": "ROE (%)",
        "roa": "ROA (%)",
        "profit_margin": "Profit Margin (%)",
        "asset_turnover": "Asset Turnover (×)",
        "rev_growth": "Revenue Growth (%)"
    })
    .style.format({
        "ROE (%)": "{:.2f}",
        "ROA (%)": "{:.2f}",
        "Profit Margin (%)": "{:.2f}",
        "Asset Turnover (×)": "{:.3f}",
        "Revenue Growth (%)": "{:.2f}"
    })
)

### 4.4 — Result Output: Charts

> Five chart types are generated below.  
> All monetary values are in **USD millions** (as reported in Compustat).  
> All ratios use the conventions defined in Section 2.3.

In [ ]:
print("[Step 10] Generating charts ...\n")

print("Chart 1 of 5 — Line Chart: Revenue Trend")
plot_line_chart(company_ratios, industry_yearly, industry_name)

print("Chart 2 of 5 — Scatter Chart: ROA vs ROE")
plot_scatter_chart(company_ratios, industry_yearly, industry_name)

print("Chart 3 of 5 — Bar Chart: Profitability Ratios")
plot_bar_chart(summary_df, industry_name)

print("Chart 4 of 5 — Box Chart: Ratio Distributions")
plot_box_chart(company_ratios, industry_name)

print("Chart 5 of 5 — Radar Chart: Five Financial Dimensions")
plot_radar_chart(summary_df, industry_name)

print("\nAll charts generated successfully.")

# Close WRDS connection to release database resources
db.close()
print("WRDS connection closed.")